# Benchmark: Slide-seq Brain Atlas

Slide-seq spatial transcriptomics data from a mouse brain atlas (101+ pucks).
Data is downloaded from [https://docs.braincelldata.org/downloads/index.html], which contains the link to google drive: 
[https://drive.google.com/drive/folders/1-3eLeLQxg2K2JEgxV_7eUg5JgDB6N7tg]. All_Pucks_h5ad.tar.gz file was downloded for this analysis. 

Key metadata columns per bead (from README):
- `Raw_Slideseq_X`, `Raw_Slideseq_Y` — raw slide-seq coordinates (used here)
- `CCF_X/Y/Z` — Allen CCF-registered coordinates
- `TopStruct`, `DeepCCF`, `CCF_acronym` — brain region annotations
- `IsOutsideCCF` — bead falls outside CCF boundary (use for filtering)

In [ ]:
import anndata as ad
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
DATA_DIR = Path('/lustre/scratch126/gengen/teams_v2/marks/dp31/SpatialPeeler/benchmark/All_Pucks_h5ad')

In [ ]:

## check how many pucks we have
puck_files = list(DATA_DIR.glob('*.h5ad'))
print(f'Found {len(puck_files)} puck files.')
PUCK_IDS = [f'{i:02d}' for i in range(1, 10)]   # 01 – 10

In [ ]:
def load_puck_obs(puck_id):
    """Load a single puck keeping only obs (skips the large X matrix)."""
    path = DATA_DIR / f'Puck_Num_{puck_id}.h5ad'
    tmp = ad.read_h5ad(path, backed='r')
    obs = tmp.obs.copy()
    obs['puck_id'] = puck_id
    tmp.file.close()
    return obs

pucks = {pid: load_puck_obs(pid) for pid in PUCK_IDS}
print(f'Loaded {len(pucks)} pucks')
for pid, obs in pucks.items():
    print(f'  Puck {pid}: {len(obs):>7,} beads  |  TopStruct: {obs["TopStruct"].value_counts().to_dict()}')

In [ ]:
len(pucks)

## Spatial maps

In [ ]:
def plot_puck(adata, title='', color='TopStruct', point_size=0.8):
    obs = adata.obs
    cats = sorted(obs[color].astype(str).unique(), key=lambda x: (x == 'NA', x))
    cmap = plt.colormaps['tab20'].resampled(len(cats))
    col_dict = {c: cmap(i) for i, c in enumerate(cats)}
    col_dict['NA'] = (0.82, 0.82, 0.82, 1.0)

    x = obs['Raw_Slideseq_X'].values.astype(float)
    y = obs['Raw_Slideseq_Y'].values.astype(float)
    c = [col_dict[str(v)] for v in obs[color]]

    fig, ax = plt.subplots(figsize=(5, 5))
    ax.scatter(x, y, c=c, s=point_size, linewidths=0, edgecolors='none', rasterized=True)
    ax.set_aspect('equal', 'box')
    ax.axis('off')
    ax.set_title(f'{title}  ({color}, n={len(obs):,})', fontsize=10)
    patches = [mpatches.Patch(color=col_dict[c], label=c) for c in cats]
    ax.legend(handles=patches, fontsize=7, loc='lower right', frameon=False, markerscale=4)
    plt.tight_layout()
    plt.show()


def plot_gene(adata, gene, title='', point_size=0.5, cmap='viridis', log1p=True):
    feat_to_idx = dict(zip(adata.var['features'], adata.var_names))
    if gene not in feat_to_idx:
        raise ValueError(f"'{gene}' not found in var['features'].")
    expr = adata[:, feat_to_idx[gene]].X
    if hasattr(expr, 'toarray'):
        expr = expr.toarray().flatten()
    else:
        expr = np.asarray(expr).flatten()
    if log1p:
        expr = np.log1p(expr)

    x = adata.obs['Raw_Slideseq_X'].values.astype(float)
    y = adata.obs['Raw_Slideseq_Y'].values.astype(float)

    fig, ax = plt.subplots(figsize=(5, 5))
    sc = ax.scatter(x, y, c=expr, s=point_size, cmap=cmap,
                    linewidths=0, edgecolors='none', rasterized=True)
    plt.colorbar(sc, ax=ax, fraction=0.03, pad=0.02,
                 label='log1p(counts)' if log1p else 'countscmap')
    ax.set_aspect('equal', 'box')
    ax.axis('off')
    ax.set_title(f'{title}  {gene}', fontsize=10)
    plt.tight_layout()
    plt.show()

In [ ]:
for pid in PUCK_IDS[:2]:   
    print(pid)
    obs = pucks[pid]
    obs = obs[obs['IsOutsideCCF'] == 0]
    tmp = ad.AnnData(obs=obs)   # lightweight: obs only, no X
    plot_puck(tmp, title=f'Puck {pid}')

In [ ]:
# Load full adata for puck 06 (including expression matrix), filter to tissue beads
adata06 = ad.read_h5ad(DATA_DIR / 'Puck_Num_06.h5ad')
adata06 = adata06[adata06.obs['IsOutsideCCF'] == 0].copy()
#del pucks   # free memory — working with puck 06 only from here
print(adata06)

In [ ]:
# Find the natural split point in puck 06 by looking at the Y-axis bead density
y_all = adata06.obs['Raw_Slideseq_Y'].values.astype(float)

print(min(y_all), max(y_all))
split_point = min(y_all) + (max(y_all) - min(y_all)) / 2
print(f'Natural split point in Y for puck 06: {split_point:.1f}')
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(y_all, bins=100, color='steelblue', edgecolor='none')
ax.axvline(split_point, color='red', linestyle='--', label=f'split at Y={split_point:.1f}')
ax.set_xlabel('Raw_Slideseq_Y')
ax.set_ylabel('Bead count')
ax.set_title('Puck 06 — Y coordinate distribution (two sections visible as two peaks)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
y = adata06.obs['Raw_Slideseq_Y'].values.astype(float)
split_y = split_point

adata06_top = adata06[y <= split_y].copy()
adata06_bot = adata06[y >  split_y].copy()

print(f'Top half:    {adata06_top.n_obs:,} beads')
print(f'Bottom half: {adata06_bot.n_obs:,} beads')

# flip bottom half vertically so it mirrors the top for comparison
FLIP_Y = True
y_bot = adata06_bot.obs['Raw_Slideseq_Y'].values.astype(float)
if FLIP_Y:
    adata06_bot.obs['Raw_Slideseq_Y'] = y_bot.max() + y_bot.min() - y_bot

plot_puck(adata06_top, 'Puck 06 — top', point_size=0.7)
plot_puck(adata06_bot, 'Puck 06 — bottom (y-flipped)', point_size=0.7)

In [ ]:
plot_puck(adata06_top, 'Puck 06 — top', color='DeepCCF', point_size=0.7)
plot_puck(adata06_bot, 'Puck 06 — bottom (y-flipped)', color='DeepCCF', point_size=0.7)

In [ ]:
genes = ['Camk2a', 'Rorb',  'Omp', 'Nrxn1', 'Mbp', 'Plp1', 'Actb', 'Snap25']
for gene in genes:
    if gene in adata06_bot.var['features'].values:
        plot_gene(adata06_bot, gene, title='Puck 06 — bottom', point_size=1, cmap='viridis_r')
        plot_gene(adata06_top, gene, title='Puck 06 — top', point_size=1, cmap='viridis_r')

In [ ]:
# Define a circular region covering ~1/5 of the puck area
# puck area ≈ pi*R^2  →  circle area = puck_area/5  →  r = R/√5
x = adata06_bot.obs['Raw_Slideseq_X'].values.astype(float)
y = adata06_bot.obs['Raw_Slideseq_Y'].values.astype(float)

cx = (x.min() + x.max()) / 2
cy = (y.min() + y.max()) / 2
puck_radius = max(x.max() - x.min(), y.max() - y.min()) / 3
circle_radius = puck_radius / np.sqrt(5)

print(f'Puck center:   ({cx:.0f}, {cy:.0f})')
print(f'Puck radius:   {puck_radius:.0f}')
print(f'Circle radius: {circle_radius:.0f}  (targets 1/5 of puck area)')

dist = np.sqrt((x - cx)**2 + (y - cy)**2)
in_circle = dist <= circle_radius
adata06_bot.obs['in_circle'] = in_circle
print(f'Beads selected: {in_circle.sum():,} / {len(in_circle):,}  ({in_circle.mean():.1%})')

# Visualize
fig, ax = plt.subplots(figsize=(5, 5))
colors = np.where(in_circle, '#e84545', '#d3d3d3')
ax.scatter(x, y, c=colors, s=0.5, linewidths=0, edgecolors='none', rasterized=True)
ax.add_patch(plt.Circle((cx, cy), circle_radius, fill=False, edgecolor='red', linewidth=1.5))
ax.set_aspect('equal', 'box')
ax.axis('off')
ax.set_title(f'Puck 06 — bottom: circle selection ({in_circle.mean():.0%} of beads)', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
import scanpy as sc

# Copy to preserve raw counts in adata06_bot for later perturbation
adata_pca = adata06_bot.copy()
sc.pp.normalize_total(adata_pca, target_sum=1e4)
sc.pp.log1p(adata_pca)
sc.pp.highly_variable_genes(adata_pca, n_top_genes=3000)
sc.pp.pca(adata_pca, n_comps=15, use_highly_variable=True)

var_explained = adata_pca.uns['pca']['variance_ratio']
for i, v in enumerate(var_explained[:5]):
    print(f'  PC{i+1}: {v:.2%} variance explained')

# PC2 loadings (column index 1, 0-based)
pc2_loadings = pd.Series(
    adata_pca.varm['PCs'][:, 1],
    index=adata_pca.var_names
)

top100_idx = pc2_loadings.abs().nlargest(100).index
top100 = adata_pca.var.loc[top100_idx, 'features'].copy()
top100_df = pd.DataFrame({
    'gene': top100.values,
    'pc2_loading': pc2_loadings[top100_idx].values
}).sort_values('pc2_loading', key=abs, ascending=False).reset_index(drop=True)

print(f'\nTop 100 genes by |PC2 loading|:')
print(top100_df.to_string())

pc2_genes = top100_df['gene'].values  # use this list for perturbation

In [ ]:
pc2_genes
### save the top 100 PC2 genes for later use in perturbation
top100_df.to_csv(DATA_DIR / 'top100_pc2_genes.csv', index=False)

In [ ]:
pc2_scores = adata_pca.obsm['X_pca'][:, 1]

x = adata06_bot.obs['Raw_Slideseq_X'].values.astype(float)
y = adata06_bot.obs['Raw_Slideseq_Y'].values.astype(float)

fig, ax = plt.subplots(figsize=(5, 5))
sc = ax.scatter(x, y, c=pc2_scores, s=0.8, cmap='RdBu_r',
                linewidths=0, edgecolors='none', rasterized=True,
                vmin=-np.percentile(np.abs(pc2_scores), 99),
                vmax= np.percentile(np.abs(pc2_scores), 99))
plt.colorbar(sc, ax=ax, fraction=0.03, pad=0.02, label='PC2 score')
ax.set_aspect('equal', 'box')
ax.axis('off')
ax.set_title('Puck 06 — bottom: PC2 scores', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
import scipy.sparse as sp

# Always start from a fresh copy so re-running this cell is safe
adata06_case = adata06_bot.copy()

circle_idx = np.where(adata06_case.obs['in_circle'].values)[0]
feat_to_col = {g: i for i, g in enumerate(adata06_case.var['features'])}
gene_cols   = np.array([feat_to_col[g] for g in pc2_genes if g in feat_to_col])

print(f'Perturbing {len(circle_idx):,} beads × {len(gene_cols)} genes')

# Extract submatrix from the UNMODIFIED copy — must happen before any writes to adata06_case.X
X_sub = adata06_case.X[np.ix_(circle_idx, gene_cols)]
if sp.issparse(X_sub):
    X_sub = X_sub.toarray()
X_sub = X_sub.astype(np.float32)

# lambda per gene = mean raw counts across circle beads (floored at 0.5)
CAP = False
CAP_value = 0.5
lam = X_sub.mean(axis=0)
if CAP:
    lam = np.maximum(X_sub.mean(axis=0), CAP_value)
    print(f'lam values capped at {CAP_value}')

print(f'lam range: {lam.min():.2f} – {lam.max():.2f}')
print(f'lam shape: {lam.shape}')

rng = np.random.default_rng(28)
additions = rng.poisson(lam[np.newaxis, :], size=X_sub.shape).astype(np.float32)

print(f'additions shape: {additions.shape}')

# Build sparse delta via explicit COO triplets 
row_idx = np.repeat(circle_idx, len(gene_cols))
col_idx = np.tile(gene_cols,    len(circle_idx))

delta = sp.csr_matrix(
    (additions.flatten(), (row_idx, col_idx)),
    shape=adata06_case.X.shape, dtype=np.float32
)
adata06_case.X = (adata06_case.X + delta).tocsr()

print(f'Mean counts added per (bead, gene): {additions.mean():.2f}')
print(f'Total counts added: {additions.sum():,.0f}')

# Sanity check
def col_mean(X, rows, col):
    sub = X[rows, :][:, [col]]
    return (sub.toarray() if sp.issparse(sub) else np.asarray(sub)).mean()

for g in pc2_genes[:3]:
    col = feat_to_col[g]
    print(f'  {g:12s}  ctrl={col_mean(adata06_bot.X,  circle_idx, col):.2f}'
          f'  case={col_mean(adata06_case.X, circle_idx, col):.2f}')

In [ ]:

# Scatter: ctrl vs case mean expression per gene inside the circle
# Each dot = one of the 100 sim genes; points above y=x means perturbation worked
feat_to_col_full = {g: i for i, g in enumerate(adata06_bot.var['features'])}

ctrl_means, case_means, loadings, gene_names = [], [], [], []
for g, loading in zip(top100_df['gene'], top100_df['pc2_loading']):
    if g not in feat_to_col_full:
        continue
    col = feat_to_col_full[g]
    ctrl_means.append(col_mean(adata06_bot.X,  circle_idx, col))
    case_means.append(col_mean(adata06_case.X, circle_idx, col))
    loadings.append(loading)
    gene_names.append(g)

ctrl_means = np.array(ctrl_means)
case_means = np.array(case_means)
loadings   = np.array(loadings)

fig, ax = plt.subplots(figsize=(6, 6))
sc = ax.scatter(ctrl_means, case_means, c=loadings,
                cmap='RdBu_r', s=40, linewidths=0.4,
                edgecolors='grey', zorder=3)
plt.colorbar(sc, ax=ax, label='PC2 loading', shrink=0.7)

lim = max(ctrl_means.max(), case_means.max()) * 1.1
ax.plot([0, lim], [0, lim], 'k--', lw=0.8, label='y = x (no change)')

# label top 10 genes by loading value
top_n = 10
top_idx = np.argsort(loadings)[::-1][:top_n]
for i in top_idx:
    ax.annotate(gene_names[i], (ctrl_means[i], case_means[i]),
                fontsize=7, xytext=(4, 2), textcoords='offset points')

ax.set_xlabel('Mean counts in circle — control (unperturbed)', fontsize=11)
ax.set_ylabel('Mean counts in circle — case (perturbed)',      fontsize=11)
ax.set_title('Perturbation check: per-gene mean expression inside circle\n'
             '(all points should be above y=x)', fontsize=11)
ax.legend(fontsize=9, frameon=False)
ax.set_xlim(0, lim); ax.set_ylim(0, lim)
plt.tight_layout()
plt.show()

fold_changes = case_means / np.where(ctrl_means > 0, ctrl_means, np.nan)
print(f'Genes with case > ctrl: {(case_means > ctrl_means).sum()} / {len(case_means)}')
print(f'Median fold-change inside circle: {np.nanmedian(fold_changes):.2f}x')

In [ ]:

# Scatter: ctrl vs case mean expression per gene inside the circle
# Each dot = one of the 100 sim genes; points above y=x means perturbation worked
feat_to_col_full = {g: i for i, g in enumerate(adata06_bot.var['features'])}

ctrl_means, case_means, loadings, gene_names = [], [], [], []
for g, loading in zip(top100_df['gene'], top100_df['pc2_loading']):
    if g not in feat_to_col_full:
        continue
    col = feat_to_col_full[g]
    ctrl_means.append(col_mean(adata06_bot.X,  circle_idx, col))
    case_means.append(col_mean(adata06_case.X, circle_idx, col))
    loadings.append(loading)
    gene_names.append(g)

ctrl_means = np.array(ctrl_means)
case_means = np.array(case_means)
loadings   = np.array(loadings)

fig, ax = plt.subplots(figsize=(6, 6))
sc = ax.scatter(ctrl_means, case_means, c=loadings,
                cmap='RdBu_r', s=40, linewidths=0.4,
                edgecolors='grey', zorder=3)
plt.colorbar(sc, ax=ax, label='PC2 loading', shrink=0.7)

lim = max(ctrl_means.max(), case_means.max()) * 1.1
ax.plot([0, lim], [0, lim], 'k--', lw=0.8, label='y = x (no change)')

# label top 10 genes by loading value
top_n = 10
top_idx = np.argsort(loadings)[::-1][:top_n]
for i in top_idx:
    ax.annotate(gene_names[i], (ctrl_means[i], case_means[i]),
                fontsize=7, xytext=(4, 2), textcoords='offset points')

ax.set_xlabel('Mean counts in circle — control (unperturbed)', fontsize=11)
ax.set_ylabel('Mean counts in circle — case (perturbed)',      fontsize=11)
ax.set_title('Perturbation check: per-gene mean expression inside circle\n'
             '(all points should be above y=x)', fontsize=11)
ax.legend(fontsize=9, frameon=False)
ax.set_xlim(0, lim); ax.set_ylim(0, lim)
plt.tight_layout()
plt.show()

fold_changes = case_means / np.where(ctrl_means > 0, ctrl_means, np.nan)
print(f'Genes with case > ctrl: {(case_means > ctrl_means).sum()} / {len(case_means)}')
print(f'Median fold-change inside circle: {np.nanmedian(fold_changes):.2f}x')

In [ ]:
genes_to_plot = ['Camk1d', 'Ywhab', 'Rbfox1', 'Syn2', 'Lars2']

for gene in genes_to_plot:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))

    for ax, adata, label in zip(axes,
                                 [adata06_bot, adata06_case],
                                 ['control', 'case (perturbed)']):
        feat_to_idx = dict(zip(adata.var['features'], adata.var_names))
        expr = adata[:, feat_to_idx[gene]].X
        if hasattr(expr, 'toarray'):
            expr = expr.toarray().flatten()
        else:
            expr = np.asarray(expr).flatten()
        expr = np.log1p(expr)

        x = adata.obs['Raw_Slideseq_X'].values.astype(float)
        y = adata.obs['Raw_Slideseq_Y'].values.astype(float)
        vmax = np.percentile(expr, 99)

        sc = ax.scatter(x, y, c=expr, s=0.8, cmap='magma_r',
                        vmin=0, vmax=vmax,
                        linewidths=0, edgecolors='none', rasterized=True)
        plt.colorbar(sc, ax=ax, fraction=0.03, pad=0.02, label='log1p(counts)')
        ax.set_aspect('equal', 'box')
        ax.axis('off')
        ax.set_title(f'{gene} — {label}', fontsize=10)

    plt.tight_layout()
    plt.show()

In [ ]:
OUT_DIR = Path('/lustre/scratch126/gengen/teams_v2/marks/dp31/SpatialPeeler/benchmark/generated_benchmark_data')
OUT_DIR.mkdir(exist_ok=True)

def clean_for_save(adata):
    a = adata.copy()
    a.raw = None  # raw.var also has '_index' which anndata can't write
    a.var = a.var.rename(columns={'_index': 'gene_id'})
    if a.var.index.name == '_index':
        a.var.index.name = None
    return a

for name, adata in [('adata06_bot',      adata06_bot),
                     ('adata06_bot_case', adata06_case),
                     ('adata06_top',      adata06_top)]:
    clean_for_save(adata).write_h5ad(OUT_DIR / f'{name}.h5ad')

print('Saved:')
for f in sorted(OUT_DIR.glob('*.h5ad')):
    print(f'  {f.name}  ({f.stat().st_size / 1e6:.0f} MB)')

In [ ]:
ROOT = Path('/lustre/scratch126/gengen/teams_v2/marks/dp31/SpatialPeeler')
DATA_DIR = ROOT / 'benchmark' / 'generated_benchmark_data'
adata06_top = ad.read_h5ad(DATA_DIR/'adata06_top.h5ad')
adata06_case = ad.read_h5ad(DATA_DIR/'adata06_bot_case.h5ad')
adata06_bot = ad.read_h5ad(DATA_DIR/'adata06_bot.h5ad')



genes_to_plot = ['Camk1d', 'Ywhab', 'Rbfox1', 'Syn2', 'Lars2']

for gene in genes_to_plot:
    fig, axes = plt.subplots(1, 3, figsize=(10, 4.5))

    for ax, adata, label in zip(axes,
                                 [adata06_top, adata06_case, adata06_bot],
                                 ['control (top)', 'case (bottom perturbed)', 'bottom']):
        feat_to_idx = dict(zip(adata.var['features'], adata.var_names))
        expr = adata[:, feat_to_idx[gene]].X
        if hasattr(expr, 'toarray'):
            expr = expr.toarray().flatten()
        else:
            expr = np.asarray(expr).flatten()
        expr = np.log1p(expr)

        x = adata.obs['Raw_Slideseq_X'].values.astype(float)
        y = adata.obs['Raw_Slideseq_Y'].values.astype(float)
        vmax = np.percentile(expr, 99)

        sc = ax.scatter(x, y, c=expr, s=0.8, cmap='magma_r',
                        vmin=0, vmax=vmax,
                        linewidths=0, edgecolors='none', rasterized=True)
        plt.colorbar(sc, ax=ax, fraction=0.03, pad=0.02, label='log1p(counts)')
        ax.set_aspect('equal', 'box')
        ax.axis('off')
        ax.set_title(f'{gene} — {label}', fontsize=10)

    plt.tight_layout()
    plt.show()